# Training Modeel

## Install dependencies

In [ ]:
!pip install matplotlib pandas sentence_transformers torch transformers

## Import dependencies

In [ ]:
from itertools import product
import math
import matplotlib.pyplot as plt
import os
import pandas as pd
from sentence_transformers import SentenceTransformer
import torch
from torch import nn
from torch.nn import functional as F
from torch.optim import AdamW
from torch.utils.data import DataLoader
from transformers import AutoTokenizer

print(f"Import is finished ...")

## Customised Classes

### AVDataset

In [ ]:
import pandas as pd
import torch
from torch.utils.data import Dataset


class AVDataset(Dataset):
    def __init__(self, csv_file_normal: str, csv_file_manual_stylometry: str, text_1_column="text_1", text_2_column="text_2", author_score_column="author_score", style_similarity_score_column="style_similarity_score", separation_token="[SEP]"):
        # Load CSV
        self.df_normal = pd.read_csv(csv_file_normal)
        self.df_manual_stylomtry = pd.read_csv(csv_file_manual_stylometry)

        assert len(self.df_normal) == len(self.df_manual_stylomtry), f"Mismatch between lengths of dataframes: {len(self.df_normal)} != {len(self.df_manual_stylomtry)}"

        # Columns
        self.text_1 = self.df_normal[text_1_column].astype(str).tolist()
        self.text_2 = self.df_normal[text_2_column].astype(str).tolist()
        if author_score_column in self.df_normal.columns:
            # Non-testing
            self.author_score = self.df_normal[author_score_column].astype(float).tolist()
        else:
            # Testing
            self.author_score = None
        self.style_similarity_score = self.df_normal[style_similarity_score_column].astype(float).tolist()

        # Load the whole csv file
        self.manual_stylomtries = self.df_manual_stylomtry.values

        assert len(self.df_normal) == len(self.manual_stylomtries), f"Mismatch between lengths between self.df_normal and self.manual_stylometries: {len(self.df_normal)} != {len(self.manual_stylomtries)}"

        self.separation_token = separation_token

        # Sanity check
        if self.author_score is not None:
            # Non-testing
            assert len(self.text_1) == len(self.author_score) and len(self.text_2) == len(self.author_score) and len(self.style_similarity_score) == len(self.author_score), f"Mismatch between lengths: text_1={len(self.text_1)}, text_2={len(self.text_2)}, author_score={len(self.author_score)}, style_similarity_score={len(self.style_similarity_score)}"
        else:
            # Testing
            assert len(self.text_1) == len(self.text_2) and len(self.style_similarity_score) == len(self.text_2), f"Mismatch between lengths: text_1={len(self.text_1)}, text_2={len(self.text_2)} style_similarity_score={len(self.style_similarity_score)}"

    def __len__(self):
        return len(self.text_1)

    def __getitem__(self, idx):
        """
        batch_manual_stylometry_y is literally author_score, which is the label
        """
        # Return a tuple
        combined_text = (self.text_1[idx], self.text_2[idx])
        if self.author_score is not None:
            # Non-testing
            author_score = torch.tensor(self.author_score[idx], dtype=torch.float)
        else:
            author_score = None
        style_similarity_score = torch.tensor(
            self.style_similarity_score[idx], dtype=torch.float)
        manual_stylometry = torch.tensor(self.manual_stylomtries[idx], dtype=torch.float)        

        if author_score is not None:
            # Non-testing
            return (combined_text, author_score, style_similarity_score, manual_stylometry, author_score)
        else:
            # Testing
            return (combined_text, style_similarity_score, manual_stylometry)


### MultiHeadCrossEncoder

In [ ]:
import torch
from torch import nn, functional as F
from transformers import AutoModel

"""
Return different scores, which will be used to determine the final score in the task of Author Verification
"""


class MultiHeadCrossEncoder(nn.Module):
    model_name: str
    head_weights: torch.Tensor

    def __init__(self, model_name: str, head_weights: torch.Tensor, manual_stylometry_dim: int = 171):
        super().__init__()

        self.encoder = AutoModel.from_pretrained(model_name)

        encoder_hidden_size = self.encoder.config.hidden_size

        self.shared = nn.Linear(encoder_hidden_size, encoder_hidden_size)
        self.shared_activation = nn.ReLU()

        self.head_author = nn.Linear(encoder_hidden_size, 1)
        self.head_style_similarity = nn.Linear(encoder_hidden_size, 1)
        self.head_manual_stylometry = nn.Sequential(
            nn.Linear(manual_stylometry_dim, 64), 
            nn.ReLU(), 
            nn.Linear(64, 1)
        )

        assert isinstance(
            head_weights, torch.Tensor), f"Invalid type of head_weights: {type(head_weights)}"
        assert head_weights.dim() == 1 and head_weights.numel() == 3, f"Invalid shape of head_weights (expect [3]): {head_weights.shape}"

        if sum(head_weights) == 1:
            self.head_weights = head_weights
        else:
            # Apply softmax function
            self.head_weights = F.softmax(head_weights, dim=0)

    def forward(self, input_ids, attention_mask, manual_stylometry_vector: torch.Tensor, token_type_ids=None):
        encoder_outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids
        )

        embedding_cls = encoder_outputs.last_hidden_state[:, 0]

        shared_output = self.shared_activation(self.shared(embedding_cls))

        # Individual heads
        score_author = self.head_author(shared_output)
        score_style_similarity = self.head_style_similarity(shared_output)
        score_manual_stylometry = self.head_manual_stylometry(manual_stylometry_vector)

        score_final = (self.head_weights[0] * score_author) + (self.head_weights[1] * score_style_similarity) + (self.head_weights[2] * score_manual_stylometry)

        return {
            f"score_final": score_final,
            f"score_author": score_author,
            f"score_style_similarity": score_style_similarity, 
            f"score_manual_stylometry": score_manual_stylometry
        }


## Global Variable

In [ ]:
# If prediction >= SIGMOID_THRESHOLD, then it is set to 1. 0 otherwise. 
SIGMOID_THRESHOLD = 0.5
SEPARATION_TOKEN = f"[SEP]"

# Input CSV file
CSV_NAME = f"AV_trial.csv"
# TODO: Update me if applicable
CSV_PATH = os.path.join("data", CSV_NAME)

# For generating csv file of style similarity
OUTPUT_CSV_STYLE_SIMILARITY_NAME = f"{CSV_NAME.split('.')[0]}_final.{CSV_NAME.split('.')[1]}"
OUTPUT_CSV_STYLE_SIMILARITY_PATH = os.path.join(CSV_PATH, "..", OUTPUT_CSV_STYLE_SIMILARITY_NAME)
MODEL_NAME = f"sentence-transformers/all-MiniLM-L6-v2"

# Larger batch for efficiency
BATCH_SIZE = 32
SHUFFLE_DATA = False
TRAIN_EPOCH = 10
LEARNING_RATE = 2e-5
# Number of train_epoch to log once
EPOCH_PER_LOG = 1

ROOT_PATH = os.path.dirname(os.path.dirname(os.getcwd()))

# TODO: Update me if applicable
KAGGLE_DATASET_NAME = f"authorverification8"

# TODO: Update me when applicable
CSV_NAME_TRAIN_NORMAL = f"train_final.csv"
# CSV_NAME_TRAIN_NORMAL = f"AV_trial_final.csv"
# For Kaggle working environment
CSV_PATH_TRAIN_NORMAL = os.path.join(ROOT_PATH, "kaggle", "input", "datasets", "toothless7788", KAGGLE_DATASET_NAME, CSV_NAME_TRAIN_NORMAL)

CSV_NAME_VALIDATION_NORMAL = f"dev_final.csv"
# For Kaggle working environment
CSV_PATH_VALIDATION_NORMAL = os.path.join(ROOT_PATH, "kaggle", "input", "datasets", "toothless7788", KAGGLE_DATASET_NAME, CSV_NAME_VALIDATION_NORMAL)

CSV_NAME_TEST_NORMAL = f"test_final.csv"
# For Kaggle working environment
CSV_PATH_TEST_NORMAL = os.path.join(ROOT_PATH, "kaggle", "input", "datasets", "toothless7788", KAGGLE_DATASET_NAME, CSV_NAME_TEST_NORMAL)

# TODO: Update me when applicable
CSV_NAME_TRAIN_MANUAL_STYLOMETRY = f"train_style_diff.csv"
# CSV_NAME_TRAIN_MANUAL_STYLOMETRY = f"trial_style_diff.csv"
# For Kaggle working environment
CSV_PATH_TRAIN_MANUAL_STYLOMETRY = os.path.join(ROOT_PATH, "kaggle", "input", "datasets", "toothless7788", KAGGLE_DATASET_NAME, CSV_NAME_TRAIN_MANUAL_STYLOMETRY)

CSV_NAME_VALIDATION_MANUAL_STYLOMETRY = f"val_style_diff.csv"
# For Kaggle working environment
CSV_PATH_VALIDATION_MANUAL_STYLOMETRY = os.path.join(ROOT_PATH, "kaggle", "input", "datasets", "toothless7788", KAGGLE_DATASET_NAME, CSV_NAME_VALIDATION_MANUAL_STYLOMETRY)

CSV_NAME_TEST_MANUAL_STYLOMETRY = f"test_style_diff.csv"
# For Kaggle working environment
CSV_PATH_TEST_MANUAL_STYLOMETRY = os.path.join(ROOT_PATH, "kaggle", "input", "datasets", "toothless7788", KAGGLE_DATASET_NAME, CSV_NAME_TEST_MANUAL_STYLOMETRY)

CSV_NAME_OUTPUT = f"predictions.csv"
# For Kaggle working environment
CSV_PATH_OUTPUT = os.path.join(ROOT_PATH, "kaggle", "working", CSV_NAME_OUTPUT)

CROSS_ENCODER_NAME = f"cross-encoder.pth"
# For Kaggle working environment
CROSS_ENCODER_PATH = os.path.join(ROOT_PATH, "kaggle", "working", CROSS_ENCODER_NAME)

print(f"Global variables are set ...")

## Generate Customised Input Files
- ### Style Similarity Score
- ### Stylometric Features

### Style Similarity Score

In [ ]:
print(f"Reading csv file at {CSV_PATH} ...")
    
# Load dataset
dataset_raw = AVDataset(CSV_PATH)
dataloader_raw = DataLoader(dataset_raw, batch_size=8, shuffle=False)  # Larger batch for efficiency

# Load pre-trained sentence transformer
model = SentenceTransformer(MODEL_NAME)

author_scores = []
style_similarity_scores = []
all_text_1 = []
all_text_2 = []

for (texts_1, texts_2), labels in dataloader_raw:
    # (texts_1, texts_2) is 1 batch

    # Compute embeddings
    # Shape: [batch_size, feature_num]
    embedding_1 = model.encode(texts_1, convert_to_tensor=True)
    # Shape: [batch_size, feature_num]
    embedding_2 = model.encode(texts_2, convert_to_tensor=True)

    # Cosine similarity
    # Shape: [batch_size]
    style_similarity_score = F.cosine_similarity(embedding_1, embedding_2, dim=1)

    # Save data
    style_similarity_scores += style_similarity_score.detach().cpu().tolist()
    all_text_1 += list(texts_1)
    all_text_2 += list(texts_2)
    author_scores += labels.tolist()

# Create new DataFrame
df_style_similarity = pd.DataFrame({
    "text_1": all_text_1,
    "text_2": all_text_2,
    "author_score": author_scores, 
    "style_similarity_score": style_similarity_scores
})

# Save CSV
df_style_similarity.to_csv(OUTPUT_CSV_STYLE_SIMILARITY_PATH, index=False)
print(f"Final training data saved to {OUTPUT_CSV_STYLE_SIMILARITY_PATH}")

### Stylometric Feature